In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
import os

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [3]:
import pandas as pd

column_names = [
    "target",
    "title",
    "text"
]

train_df = pd.read_csv(
    "train.txt",
    sep="\t",
    names=column_names,
    skiprows=1
)

train_df.head()

,target,title,text
0,BACKGROUND,The emergence of HIV as a chronic condition me...,NaN
1,BACKGROUND,This paper describes the design and evaluation...,NaN
2,METHODS,This study is designed as a randomised control...,NaN
3,METHODS,The intervention group will participate in the...,NaN
4,METHODS,The program is based on self-efficacy theory a...,NaN


In [4]:
print(train_df.shape)
train_df.columns

(1691280, 3)


Index(['target', 'title', 'text'], dtype='object')

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1691280 entries, 0 to 1691279
Data columns (total 3 columns):
 #   Column  Dtype  
---  ------  -----  
 0   target  object 
 1   title   object 
 2   text    float64
dtypes: float64(1), object(2)
memory usage: 38.7+ MB


In [6]:
train_df.isnull().sum()

,0
target,0
title,134181
text,1691280


In [7]:
train_df = train_df.drop(columns=["text"])

train_df = train_df.dropna()

train_df.reset_index(drop=True, inplace=True)

print(train_df.shape)

train_df.head()

(1557099, 2)


,target,title
0,BACKGROUND,The emergence of HIV as a chronic condition me...
1,BACKGROUND,This paper describes the design and evaluation...
2,METHODS,This study is designed as a randomised control...
3,METHODS,The intervention group will participate in the...
4,METHODS,The program is based on self-efficacy theory a...


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!


In [9]:
train_df = train_df.head(5000)
print(train_df.shape)

(5000, 2)


In [10]:
embeddings = model.encode(
    train_df["title"].tolist(),
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings Shape: (5000, 384)


In [11]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

print("FAISS Index Created Successfully!")

FAISS Index Created Successfully!


In [12]:
def search_papers(query, k=5):

    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    print("="*60)
    print("Search Query:", query)
    print("="*60)

    for i, idx in enumerate(indices[0]):
        print(f"\nResult {i+1}")
        print("Category :", train_df.iloc[idx]["target"])
        print("Paper :", train_df.iloc[idx]["title"])

In [13]:
search_papers("deep learning for brain tumor detection")

Search Query: deep learning for brain tumor detection

Result 1
Category : CONCLUSIONS
Paper : An approach to lung cancer staging with PET-CT improves discrimination between N0-1 and N2-3 .

Result 2
Category : CONCLUSIONS
Paper : Our results are the basis for dose-escalation studies to improve local tumour control .

Result 3
Category : METHODS
Paper : Intermediate-risk patients were all other patients with local or regional tumors .

Result 4
Category : RESULTS
Paper : A total of 189 patients were recruited , 98 in the PET-CT group and 91 in the CWU group .

Result 5
Category : RESULTS
Paper : Cox regression models that took TNM staging or the residual tumor classification and tumor site into account also found significant differences at 10 years .


In [14]:
print("Dataset Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns)

print("\nCategories:")
print(train_df["target"].unique())

print("\nNumber of Categories:")
print(train_df["target"].nunique())

print("\nCategory Counts:")
print(train_df["target"].value_counts())

Dataset Shape: (5000, 2)

Columns:
Index(['target', 'title'], dtype='object')

Categories:
['BACKGROUND' 'METHODS' 'CONCLUSIONS' 'RESULTS' 'OBJECTIVE']

Number of Categories:
5

Category Counts:
target
RESULTS        1728
METHODS        1648
CONCLUSIONS     787
BACKGROUND      445
OBJECTIVE       392
Name: count, dtype: int64


In [15]:
train_df["text_length"] = train_df["title"].apply(len)

train_df["text_length"].describe()

,text_length
count,5000.000000
mean,150.789200
std,77.524262
min,2.000000
25%,96.000000
50%,139.000000
75%,191.000000
max,790.000000


In [16]:
train_df.head()

,target,title,text_length
0,BACKGROUND,The emergence of HIV as a chronic condition me...,226
1,BACKGROUND,This paper describes the design and evaluation...,160
2,METHODS,This study is designed as a randomised control...,176
3,METHODS,The intervention group will participate in the...,90
4,METHODS,The program is based on self-efficacy theory a...,195


In [17]:
print("Longest Title:")
print(train_df.loc[train_df["text_length"].idxmax()]["title"])

print("\nShortest Title:")
print(train_df.loc[train_df["text_length"].idxmin()]["title"])

Longest Title:
Validation tests indicate that the 2-point LSS model developed for the reference formulation predicts accurately ( R2 > 0.90 ) : ( a ) the individual AUC0-72 for the test formulation in the same group of volunteers ; ( b ) the individual AUC0-72 for the same reference formulation in another bioequivalence study in Brazilian volunteers ; ( c ) the average AUC0-72 reported in seven additional international studies performed under protocols similar to the present investigation ; ( d ) the individual AUC0-72 corresponding to concentration data points provided by a first-order compartmental pharmacokinetic model , when the relative values of either the absorption rate ( Kabs ) or the bioavailability ( F ) model parameters were set at 0.85 or 0.6 , of their respective original values .

Shortest Title:
I.


In [18]:
import numpy as np

np.save("paper_embeddings.npy", embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [19]:
faiss.write_index(index, "paper_faiss.index")

print("FAISS Index saved successfully!")

FAISS Index saved successfully!


In [20]:
import os

print(os.listdir())

['.config', 'test.txt', 'paper_faiss.index', 'dev.txt', 'paper_embeddings.npy', 'train.txt', 'sample_data']


In [21]:
def search_papers(query, k=5):

    # Convert query into embedding
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    # Search similar papers
    distances, indices = index.search(query_embedding, k)

    print("=" * 80)
    print("🔍 Medical Abstract Search Results")
    print("=" * 80)
    print(f"Search Query : {query}")
    print(f"Top {k} Results Found\n")

    for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), start=1):

        print(f"📄 Result {rank}")
        print(f"📂 Category        : {train_df.iloc[idx]['target']}")
        print(f"📝 Abstract        : {train_df.iloc[idx]['title']}")
        print(f"⭐ Similarity Score : {score:.4f}")
        print("-" * 80)

In [22]:
search_papers("lung cancer")

🔍 Medical Abstract Search Results
Search Query : lung cancer
Top 5 Results Found

📄 Result 1
📂 Category        : METHODS
📝 Abstract        : A randomised clinical trial was conducted including patients with a verified diagnosis of non-small cell lung cancer , who were considered operable .
⭐ Similarity Score : 0.6426
--------------------------------------------------------------------------------
📄 Result 2
📂 Category        : BACKGROUND
📝 Abstract        : Correct mediastinal staging is a cornerstone in the treatment of patients with non-small cell lung cancer .
⭐ Similarity Score : 0.5743
--------------------------------------------------------------------------------
📄 Result 3
📂 Category        : CONCLUSIONS
📝 Abstract        : An approach to lung cancer staging with PET-CT improves discrimination between N0-1 and N2-3 .
⭐ Similarity Score : 0.5210
--------------------------------------------------------------------------------
📄 Result 4
📂 Category        : BACKGROUND
📝 Abstract  

In [23]:
search_papers("brain tumor")

🔍 Medical Abstract Search Results
Search Query : brain tumor
Top 5 Results Found

📄 Result 1
📂 Category        : METHODS
📝 Abstract        : Twenty-nine had lesions located in the head-and-neck area .
⭐ Similarity Score : 0.4965
--------------------------------------------------------------------------------
📄 Result 2
📂 Category        : BACKGROUND
📝 Abstract        : Ablative surgery of oropharyngeal tumors frequently leads to defects in the speech organs , resulting in impairment of speech up to the point of unintelligibility .
⭐ Similarity Score : 0.4089
--------------------------------------------------------------------------------
📄 Result 3
📂 Category        : METHODS
📝 Abstract        : Intermediate-risk patients were all other patients with local or regional tumors .
⭐ Similarity Score : 0.4082
--------------------------------------------------------------------------------
📄 Result 4
📂 Category        : METHODS
📝 Abstract        : Low-risk patients were defined as those with

In [24]:
search_papers("covid vaccine")

🔍 Medical Abstract Search Results
Search Query : covid vaccine
Top 5 Results Found

📄 Result 1
📂 Category        : OBJECTIVE
📝 Abstract        : Safety and reactogenicity of a new heptavalent DTPw-HBV/Hib-MenAC ( diphtheria , tetanus , whole cell pertussis-hepatitis B virus/Haemophilus influenzae type b-Neisseria meningitidis serogroups A and C ) vaccine was compared with a widely used pentavalent DTPw-HBV/Hib vaccine .
⭐ Similarity Score : 0.5111
--------------------------------------------------------------------------------
📄 Result 2
📂 Category        : METHODS
📝 Abstract        : The first regimen given to 43 subjects , consisted of intradermal administration of 0.2 ml of vaccine at 2 sites on days 0 , 3 and 7 and at one site on days 28 and 90 .
⭐ Similarity Score : 0.5044
--------------------------------------------------------------------------------
📄 Result 3
📂 Category        : METHODS
📝 Abstract        : The second regimen , given to 39 subjects , consisted of intradermal ad